# Inference With Trained Model

Edit the config cell, load a trained checkpoint, run one conditional sample, and compare glossy input, predicted diffuse, ground-truth diffuse, and residuals.

This notebook supports both:
- new checkpoints saved by `Run_Training.py`
- legacy raw `model.pth` state dict files saved by `Trainer.py`

For legacy raw state dicts, set the manual override fields in the config cell.


In [ ]:
from pathlib import Path
from pprint import pprint
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch


def find_specdiff_dir() -> Path:
    candidates = [
        Path.cwd(),
        Path.cwd() / "Specular-Highlights" / "SpecDiff",
        Path.cwd().parent,
    ]
    for candidate in candidates:
        if (candidate / "Run_Training.py").exists():
            return candidate.resolve()
    raise FileNotFoundError("Could not locate SpecDiff/Run_Training.py from the current working directory.")


SPECDIFF_DIR = find_specdiff_dir()
PROJECT_ROOT = SPECDIFF_DIR.parent
if str(SPECDIFF_DIR) not in sys.path:
    sys.path.insert(0, str(SPECDIFF_DIR))

import Run_Training as rt
import Diffuser as diff
import ParamDiffuser as pdiff


def checkpoint_candidates(limit: int = 20) -> list[Path]:
    roots = [
        SPECDIFF_DIR / "models",
        SPECDIFF_DIR / "checkpoints",
        PROJECT_ROOT / "models",
        PROJECT_ROOT / "checkpoints",
    ]
    found = []
    seen = set()
    for root in roots:
        if not root.exists():
            continue
        for path in sorted(root.rglob("*.pth")):
            resolved = path.resolve()
            if resolved in seen:
                continue
            seen.add(resolved)
            found.append(resolved)
    return found[:limit]


def checkpoint_is_tensor_dict(value) -> bool:
    return isinstance(value, dict) and bool(value) and all(torch.is_tensor(v) for v in value.values())


def select_state_dict(checkpoint, prefer_ema: bool = True):
    if checkpoint_is_tensor_dict(checkpoint):
        return checkpoint, "raw_state_dict"

    if not isinstance(checkpoint, dict):
        raise TypeError(f"Unsupported checkpoint type: {type(checkpoint)!r}")

    if prefer_ema and checkpoint_is_tensor_dict(checkpoint.get("ema_model_state_dict")):
        return checkpoint["ema_model_state_dict"], "ema_model_state_dict"
    if checkpoint_is_tensor_dict(checkpoint.get("model_state_dict")):
        return checkpoint["model_state_dict"], "model_state_dict"
    if checkpoint_is_tensor_dict(checkpoint.get("state_dict")):
        return checkpoint["state_dict"], "state_dict"

    raise KeyError("Could not find a usable state dict in the checkpoint.")


def checkpoint_value(checkpoint, key: str, override, default=None):
    if override is not None:
        return override
    if isinstance(checkpoint, dict):
        return checkpoint.get(key, default)
    return default


def chw_to_display(chw: torch.Tensor) -> np.ndarray:
    array = chw.detach().cpu().permute(1, 2, 0).float().numpy()
    if array.min() < 0.0:
        scale = max(abs(float(array.min())), abs(float(array.max())), 1e-6)
        array = 0.5 + array / (2.0 * scale)
    return np.clip(array, 0.0, 1.0)


def tensor_stats(name: str, tensor: torch.Tensor) -> dict[str, object]:
    return {
        "name": name,
        "shape": tuple(tensor.shape),
        "min": float(tensor.min()),
        "max": float(tensor.max()),
        "mean": float(tensor.mean()),
        "std": float(tensor.std(unbiased=False)),
        "all_finite": bool(torch.isfinite(tensor).all()),
    }


print(f"SpecDiff dir: {SPECDIFF_DIR}")


In [ ]:
# Paste your trained checkpoint path here.
CHECKPOINT_PATH = None

# Checkpoint loading.
PREFER_EMA = True
DEVICE = "auto"  # "auto", "cpu", "cuda"

# Dataset sample for inference.
DATASET_SOURCE = "psd"  # "psd" or "local"
PSD_ROOT = "/share/lcn_projects/z0058vfs/project_hl.PSD_Dataset"
SPLIT = "train"
GLOSSY_DIR = None
DIFFUSE_DIR = None
SAMPLE_INDEX = 0

# Sampling.
SHOW_PROGRESS = False
USE_DDIM_FOR_E = True
DDIM_SKIP_STEPS = 2

# Manual overrides for legacy raw state_dict checkpoints.
BACKBONE_MODULE_OVERRIDE = None
MODEL_NAME_OVERRIDE = None
IMAGE_SIZE_OVERRIDE = None
NOISE_STEPS_OVERRIDE = None
DEPTH_OVERRIDE = None
PARAMETERIZATION_OVERRIDE = None
TARGET_SCALE_OVERRIDE = None


In [ ]:
if CHECKPOINT_PATH is None:
    print("Set CHECKPOINT_PATH in the config cell. Candidate .pth files:")
    for candidate in checkpoint_candidates():
        print(candidate)
    raise ValueError("CHECKPOINT_PATH is not set.")

checkpoint_path = Path(CHECKPOINT_PATH).expanduser().resolve()
checkpoint = torch.load(checkpoint_path, map_location="cpu")
state_dict, state_key = select_state_dict(checkpoint, prefer_ema=PREFER_EMA)

backbone_module = checkpoint_value(checkpoint, "backbone_module", BACKBONE_MODULE_OVERRIDE, default="SDEBackbone")
model_name = checkpoint_value(checkpoint, "model_name", MODEL_NAME_OVERRIDE, default="unetwithtransformer")
image_size = int(checkpoint_value(checkpoint, "image_size", IMAGE_SIZE_OVERRIDE, default=32))
noise_steps = int(checkpoint_value(checkpoint, "noise_steps", NOISE_STEPS_OVERRIDE, default=200))
depth = int(checkpoint_value(checkpoint, "depth", DEPTH_OVERRIDE, default=4))
parameterization = str(checkpoint_value(checkpoint, "parameterization", PARAMETERIZATION_OVERRIDE, default="e"))
target_scale = float(checkpoint_value(checkpoint, "target_scale", TARGET_SCALE_OVERRIDE, default=1.0))
if target_scale == 0.0:
    raise ValueError("target_scale must be non-zero for inference.")

device = rt.resolve_device(DEVICE)
model = rt.build_model(
    backbone_module=backbone_module,
    model_name=model_name,
    image_size=image_size,
    noise_steps=noise_steps,
    depth=depth,
).to(device)
model.load_state_dict(state_dict)
model.eval()

if parameterization == "e":
    sampler = diff.CosSchDiffuser(steps=noise_steps, device=device)
    sampler_name = "Diffuser.CosSchDiffuser"
else:
    sampler = pdiff.CosSchDiffuser(steps=noise_steps, device=device)
    sampler_name = "ParamDiffuser.CosSchDiffuser"

print(f"checkpoint: {checkpoint_path}")
print(f"state key: {state_key}")
print(f"device: {device}")
print(f"backbone: {backbone_module}")
print(f"model: {model_name}")
print(f"image size: {image_size}")
print(f"noise steps: {noise_steps}")
print(f"depth: {depth}")
print(f"parameterization: {parameterization}")
print(f"target scale: {target_scale}")
print(f"sampler: {sampler_name}")


In [ ]:
loader, data_info = rt.build_data_loader(
    dataset_source=DATASET_SOURCE,
    psd_root=Path(PSD_ROOT).expanduser().resolve() if PSD_ROOT else None,
    split=SPLIT,
    glossy_dir=GLOSSY_DIR,
    diffuse_dir=DIFFUSE_DIR,
    image_size=image_size,
    batch_size=1,
    shuffle=False,
    num_workers=0,
)

dataset = loader.dataset
sample = dataset[SAMPLE_INDEX]
condition = sample[:3].unsqueeze(0)
diffuse_gt = sample[3:].unsqueeze(0)
residual_gt = diffuse_gt - condition

print("Loader info:")
pprint(data_info)
print(f"dataset size: {len(dataset)}")
print(f"sample index: {SAMPLE_INDEX}")
pprint(tensor_stats("condition", condition))
pprint(tensor_stats("diffuse_gt", diffuse_gt))
pprint(tensor_stats("residual_gt", residual_gt))


In [ ]:
condition_device = condition.to(device)
diffuse_gt_device = diffuse_gt.to(device)
residual_gt_device = residual_gt.to(device)

with torch.no_grad():
    if parameterization == "e":
        sampled_scaled_residual = sampler.sample_from_noise(
            model,
            condition_device,
            show_progress=SHOW_PROGRESS,
            ddim=USE_DDIM_FOR_E,
            skip_steps=DDIM_SKIP_STEPS,
        )
    else:
        sampled_scaled_residual = sampler.sample_from_noise(
            model,
            condition_device,
            parameterization=parameterization,
            show_progress=SHOW_PROGRESS,
        )

predicted_residual = sampled_scaled_residual / target_scale
predicted_diffuse = condition_device + predicted_residual
predicted_diffuse_clamped = predicted_diffuse.clamp(0.0, 1.0)

residual_error = predicted_residual - residual_gt_device
diffuse_error = predicted_diffuse - diffuse_gt_device
diffuse_error_clamped = predicted_diffuse_clamped - diffuse_gt_device

metrics = {
    "residual_mae": float(residual_error.abs().mean()),
    "residual_rmse": float(torch.sqrt((residual_error ** 2).mean())),
    "diffuse_mae": float(diffuse_error.abs().mean()),
    "diffuse_mae_clamped": float(diffuse_error_clamped.abs().mean()),
}
metrics


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 9))

panels = [
    ("glossy / condition", condition[0]),
    ("ground truth diffuse", diffuse_gt[0]),
    ("pred diffuse", predicted_diffuse_clamped[0].cpu()),
    ("abs diffuse error", diffuse_error_clamped.abs()[0].cpu()),
    ("ground truth residual", residual_gt[0]),
    ("pred residual", predicted_residual[0].cpu()),
    ("abs residual error", residual_error.abs()[0].cpu()),
    ("pred scaled residual", sampled_scaled_residual[0].cpu()),
]

for axis, (title, tensor) in zip(axes.flat, panels):
    axis.imshow(chw_to_display(tensor))
    axis.set_title(title)
    axis.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
summary = {
    "checkpoint": str(checkpoint_path),
    "state_key": state_key,
    "dataset_source": DATASET_SOURCE,
    "split": SPLIT,
    "sample_index": SAMPLE_INDEX,
    "parameterization": parameterization,
    "target_scale": target_scale,
    "sampler": sampler_name,
    **metrics,
}
pprint(summary)
